# State of the art: R0 by the next-generation matrix and the derivation of the final size

- **Level:** 3, State of the art
- **Prerequisites:** notebooks 01 and 02 of this topic
- **Reading time:** about 30 minutes

## Objectives

By the end of this notebook you can:

- compute $R_0$ as the spectral radius of the next-generation matrix and compare it with $\beta/\gamma$;
- derive the final-size relation from the model equations, including the initial infection, and check the derivation on the engine's trajectory;
- explain the difference between the engine's attack rate and the relation of notebook 02, and remove it by extrapolating in the step.

## Setup

The notebook environment is pinned in `notebooks/requirements.txt` and built with `scripts/setup.sh --notebooks` (`scripts/setup.ps1 -Notebooks` on Windows). The packages this notebook uses, with the exact pins:

```bash
pip install numpy==2.4.6 matplotlib==3.11.2
```

This example product publishes no package of its own: its method lives in `data-pipeline/pipeline/`, so the notebook imports that code by its path. A product imports its own PyPI package instead (`pip install <package>==<version>`).

The next cell imports the tools and prints their versions.

In [1]:
# Print the versions of the tools this notebook uses, to compare them with notebooks/requirements.txt
import importlib.metadata as md
import json

import numpy as np
import matplotlib.pyplot as plt

for name in ("numpy", "matplotlib"):
    print(name, md.version(name))

numpy 2.4.6
matplotlib 3.11.2


The engine is imported from `data-pipeline/` by its path, as in topic 01.

In [2]:
# The notebook sits three folders below the repository root; the pipeline's modules are imported by their path
import sys
from pathlib import Path

ROOT = Path("../../..").resolve()
sys.path.insert(0, str(ROOT / "data-pipeline"))
from pipeline.model.sir import simulate
from pipeline.io.schema import SIRParams
from pipeline.oracles import final_size

print(simulate.__module__, SIRParams.__module__)

pipeline.model.sir pipeline.io.schema


## Data

The parameters of the five baked cases, the same as in the earlier notebooks, so that every comparison below is on the same cases as level 2.

- Owner: this repository (the product template's example), https://github.com/fsantibanezleal/CAOS_PRODUCT_TEMPLATE.
- Licence class: synthetic data produced by the pipeline from the parameters in each manifest (`real_or_synthetic` is `synthetic`); no third-party dataset, so no dataset licence applies, and the repository's MIT licence covers the artifacts.
- Files read: `data/derived/manifests/index.json` and `data/derived/manifests/<case>.json`.

The next cell defines the readers and lists the cases.

In [3]:
# Two readers of the committed artifacts: a missing file stops the notebook and names its path
def read_json(relative):
    return json.loads((ROOT / "data" / "derived" / relative).read_text(encoding="utf-8"))


def params_of(case_id):
    p = read_json(f"manifests/{case_id}.json")["params"]
    return SIRParams(case_id, p["beta"], p["gamma"], p["N"], p["I0"], p["days"])


CASES = [c["case_id"] for c in read_json("manifests/index.json")["cases"]]
OUTBREAKS = [c for c in CASES if c.startswith("EX")]
SUPERCRITICAL = [c for c in OUTBREAKS if params_of(c).beta > params_of(c).gamma]
print(CASES)
print("cases that can take off:", SUPERCRITICAL)

['CTRL_degenerate', 'EX01_subcritical', 'EX02_epidemic', 'EX03_fast_burn', 'EX04_slow_spread']
cases that can take off: ['EX02_epidemic', 'EX03_fast_burn', 'EX04_slow_spread']


## Method

What level 2 established, in two lines: $R_0 = \beta / \gamma$ (notebook 02 of topic 01) and the final size $z$ solves $z = 1 - e^{-R_0 z}$ (notebook 02 of this topic), checked against the engine's Forward Euler run at a quarter-day step. This notebook asks whether the state of the art gives the same two answers and what it adds.

## State of the art

### R0 by the next-generation matrix

For a compartmental model, van den Driessche and Watmough (2002) split the equations of the infected compartments into the rate of new infections $\mathcal{F}$ and the rate of transitions $\mathcal{V}$ (recovery, death), linearise both at the disease-free state to the matrices $F$ and $V$, and define $R_0$ as the spectral radius of the next-generation matrix $F V^{-1}$. Their condition is that the disease-free state is stable when $R_0 < 1$. For the SIR model there is one infected compartment, so $F = [\beta]$ and $V = [\gamma]$ are $1 \times 1$ and $F V^{-1} = [\beta / \gamma]$. The cell computes it as the largest absolute eigenvalue, as for a larger model, and also the growth rate of the infected compartment at the disease-free state, the eigenvalue of $F - V$.

In [4]:
# R0 as the spectral radius of F V^-1, and the growth rate of the infected compartment as the eigenvalue of F - V
for case_id in CASES:
    p = params_of(case_id)
    F = np.array([[p.beta]])
    V = np.array([[p.gamma]])
    r0_ngm = float(np.max(np.abs(np.linalg.eigvals(F @ np.linalg.inv(V)))))
    growth = float(np.max(np.linalg.eigvals(F - V).real))
    print(f"{case_id:18s} R0 (matrix)={r0_ngm:.4f}  beta/gamma={p.beta / p.gamma:.4f}  growth rate={growth:+.3f}  R0>1: {r0_ngm > 1}  growth>0: {growth > 0}")

CTRL_degenerate    R0 (matrix)=2.0000  beta/gamma=2.0000  growth rate=+0.200  R0>1: True  growth>0: True
EX01_subcritical   R0 (matrix)=0.7200  beta/gamma=0.7200  growth rate=-0.070  R0>1: False  growth>0: False
EX02_epidemic      R0 (matrix)=2.7500  beta/gamma=2.7500  growth rate=+0.350  R0>1: True  growth>0: True
EX03_fast_burn     R0 (matrix)=6.0000  beta/gamma=6.0000  growth rate=+1.000  R0>1: True  growth>0: True
EX04_slow_spread   R0 (matrix)=1.2000  beta/gamma=1.2000  growth rate=+0.050  R0>1: True  growth>0: True


The stability condition can be checked on the engine. Run each case for its first two days with a small step and estimate the growth rate of the infected people, $\ln(I(2)/I(0))/2$, to compare it with $\beta - \gamma$. They agree only approximately, because the susceptible pool is already a little smaller than $N$ after two days; the cell prints both.

In [5]:
# Growth rate of the infected people over the first two days of the engine's run, against beta - gamma
for case_id in OUTBREAKS:
    p = params_of(case_id)
    short = SIRParams(p.case_id, p.beta, p.gamma, p.N, p.I0, 2)
    r = simulate(short, dt=0.0625)
    measured = float(np.log(r.I[-1] / r.I[0]) / 2.0)
    print(f"{case_id:18s} measured={measured:+.4f}  beta-gamma={p.beta - p.gamma:+.4f}")

EX01_subcritical   measured=-0.0703  beta-gamma=-0.0700
EX02_epidemic      measured=+0.3458  beta-gamma=+0.3500
EX03_fast_burn     measured=+0.9662  beta-gamma=+1.0000
EX04_slow_spread   measured=+0.0497  beta-gamma=+0.0500


### The final size, derived

Miller (2012) gives a closed derivation of the final size. Here it is for the SIR system, from its equations, keeping the initial infection. Dividing the first by the third equation removes time: $dS/dR = -(\beta/\gamma) S/N = -R_0 S/N$. Integrating from the start, where $R = 0$ and $S = S_0$:

$$S = S_0 \, e^{-R_0 R / N}$$

At the end of the outbreak $I = 0$, so $S_\infty = N - R_\infty$. With $z = R_\infty / N$ and $i_0 = I_0 / N$, where $S_0 = N (1 - i_0)$:

$$1 - z = (1 - i_0) \, e^{-R_0 z}$$

When the initial infection is negligible ($i_0 \to 0$) this is the relation of level 2, $z = 1 - e^{-R_0 z}$. First the cell checks the intermediate result on the engine's trajectory: the quantity $S \, e^{R_0 R / N}$ should stay constant. Forward Euler only keeps it up to its own error, so the cell shows the drift at the engine's step and at a quarter of it.

In [6]:
# Largest relative drift of S exp(R0 R / N) along the engine's run, at two steps, for the epidemic case
p = params_of("EX02_epidemic")
r0 = p.beta / p.gamma
for h in (0.25, 0.0625):
    r = simulate(p, dt=h)
    invariant = np.array(r.S) * np.exp(r0 * np.array(r.R) / p.N)
    drift = float(np.max(np.abs(invariant / invariant[0] - 1.0)))
    print(f"dt={h:6.4f}  largest relative drift={drift:.5f}")

dt=0.2500  largest relative drift=0.03143
dt=0.0625  largest relative drift=0.00772


Now solve the derived relation, with the initial infection, with the same `final_size` of `data-pipeline/pipeline/oracles.py` (its second argument is the initial infected fraction), for every case with $R_0 > 1$, and put three numbers side by side: the level 2 relation, the derived one and the engine's attack rate at the engine's step. The slow-spread case is marked, because its outbreak has not ended at the horizon.

In [7]:
# Solve 1 - z = (1 - i0) exp(-R0 z) and set it beside the level 2 relation and the engine's attack rate
print(f"{'case':18s} {'level 2':>9s} {'derived':>9s} {'engine':>9s}  ended in the horizon")
for case_id in SUPERCRITICAL:
    q = params_of(case_id)
    r = simulate(q)
    ended = r.I[-1] / q.N < 1e-6
    print(f"{case_id:18s} {final_size(q.beta / q.gamma):9.5f} {final_size(q.beta / q.gamma, q.I0 / q.N):9.5f} {r.attack_rate:9.5f}  {ended}")

case                 level 2   derived    engine  ended in the horizon
EX02_epidemic        0.92044   0.92049   0.92365  True
EX03_fast_burn       0.99748   0.99749   0.99821  True
EX04_slow_spread     0.31370   0.31563   0.29928  False


The remaining gap between the derived relation and the engine should be the first-order error of the step. If the error is $c \, h$ for a step $h$, then $2 A(h/2) - A(h)$ cancels it, where $A(h)$ is the attack rate at step $h$ (Hairer, Norsett and Wanner, 1993, give the first order). The cell applies this to the epidemic case, from the steps 1/4, 1/8 and 1/16 of a day, and compares it with the derived relation.

In [8]:
# Remove the first-order error of the step by extrapolation and compare with the derived final size
p = params_of("EX02_epidemic")
attack = {h: simulate(p, dt=h).attack_rate for h in (0.25, 0.125, 0.0625)}
extrapolated = [2 * attack[0.125] - attack[0.25], 2 * attack[0.0625] - attack[0.125]]
target = final_size(p.beta / p.gamma, p.I0 / p.N)
print("derived final size:", round(target, 6))
for h, value in attack.items():
    print(f"dt={h:6.4f}  engine={value:.6f}  gap={value - target:+.6f}")
print("extrapolated (0.25 and 0.125):", round(extrapolated[0], 6), " (0.125 and 0.0625):", round(extrapolated[1], 6))

derived final size: 0.920487
dt=0.2500  engine=0.923654  gap=+0.003167
dt=0.1250  engine=0.922060  gap=+0.001572
dt=0.0625  engine=0.921271  gap=+0.000783
extrapolated (0.25 and 0.125): 0.920465  (0.125 and 0.0625): 0.920482


## Results

- The next-generation matrix gives the same $R_0$ as $\beta/\gamma$ for every case, as it must for one infected compartment, and the sign of the growth rate of the infected compartment ($\beta - \gamma$) matches the side of 1 on which $R_0$ lies: this is the stability condition of van den Driessche and Watmough (2002). The engine's measured growth over two days is a little below $\beta - \gamma$, because the susceptible pool has already started to shrink. What the matrix adds is generality: for a model with several infected compartments the same two lines (build $F$ and $V$, take the spectral radius) give $R_0$, where $\beta/\gamma$ has no counterpart. This model does not need it, and the engine does not implement it.
- The quantity $S \, e^{R_0 R / N}$ is constant in the exact solution and drifts along the engine's run by a few percent at the engine's step; at a quarter of the step the drift is about four times smaller. The derivation holds, and the drift is the Forward Euler error.
- The derived relation, with the initial infection, differs from level 2 by 0.00005 for the epidemic case and 0.000002 for the fast burn. The engine's attack rate lies above both and approaches the derived value as the step shrinks, and the extrapolated values agree with it within 0.00003 (within 0.00001 for the finer pair of steps). So the gap seen in level 2 is the first-order error of the engine's quarter-day step, not a flaw of the relation. For the slow-spread case the outbreak has not ended, so none of the relations applies to its attack rate yet.
- What these runs do not show is anything about models with more compartments: the example engine has only three.

## Summary

- State of the art 1: $R_0$ as the spectral radius of $F V^{-1}$ reduces to $\beta/\gamma$ here and generalises to larger models.
- State of the art 2: the final size derived from the equations, with the initial infection, explains the engine's small excess over the level 2 relation as Forward Euler error.
- Both agree with level 2 on the same cases; neither changes the engine.

## References

- van den Driessche, P. and Watmough, J. (2002). Reproduction numbers and sub-threshold endemic equilibria for compartmental models of disease transmission. Mathematical Biosciences. https://doi.org/10.1016/S0025-5564(02)00108-6
- Miller, J. C. (2012). A note on the derivation of epidemic final sizes. Bulletin of Mathematical Biology. https://doi.org/10.1007/s11538-012-9749-6
- Kermack, W. O. and McKendrick, A. G. (1927). A contribution to the mathematical theory of epidemics. Proceedings of the Royal Society A. https://doi.org/10.1098/rspa.1927.0118
- Hairer, E., Norsett, S. P. and Wanner, G. (1993). Solving Ordinary Differential Equations I: Nonstiff Problems. Springer. https://doi.org/10.1007/978-3-540-78862-1
- NumPy, https://pypi.org/project/numpy/ (pinned in `notebooks/requirements.txt`).
- Matplotlib, https://pypi.org/project/matplotlib/ (pinned in `notebooks/requirements.txt`).
- The engine and the cases of this repository, https://github.com/fsantibanezleal/CAOS_PRODUCT_TEMPLATE (`data-pipeline/pipeline/model/sir.py`).

What each work is credited with is what the research basis of the SDD (`docs/design/SDD.md`) records; the text of their pages was not re-read, only their Crossref records.